# MLP-VaR Softplus: SiLU conservador - 2010-2024, w=1

Prueba a partir del mejor candidato actual (`dynamic_silu_layernorm`) sin modificar la funcion de perdida ni el target.

Cambios frente al SiLU anterior:

- menos features dinamicas para reducir picos de VaR;
- red mas pequena `64-32`;
- `Dropout(0.05)`;
- `Adam(weight_decay=1e-4)`;
- salida `Softplus` y perdida pinball identicas sobre `target` en escala original con `w=1`.


In [1]:
import copy
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import chi2
from torch.utils.data import DataLoader, TensorDataset

try:
    from tqdm.auto import tqdm
except ModuleNotFoundError:
    def tqdm(iterable, *args, **kwargs):
        return iterable

torch.set_num_threads(1)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

c:\Users\trgglg\OneDrive - gmv.com\projects_w\TFG-GONZALO-LOPEZ-GARCIA-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = Path("../data")
if not DATA_DIR.exists():
    DATA_DIR = Path("data")
DATASET_PATH = DATA_DIR / "dataset_tfg.pkl"

ALPHA = 0.99
SIG = 0.05
W_LOSS = 1
WINDOW = 900
RETRAIN_EVERY = 1
SEED = 42
WEIGHT_DECAY = 1e-4
EVAL_START = "2010-01-01"
EVAL_END = "2024-12-31"

EXPERIMENTS = [
    {
        "key": "silu_conservative",
        "label": "Softplus + conservative dynamic SiLU LayerNorm",
        "architecture": "silu_conservative",
        "prefix": "softplus_silu_conservative_2010_2024_w1",
        "pred_file": "nn_softplus_silu_conservative_2010_2024_w1.csv",
    },
]


## Carga de datos y features dinamicas

In [3]:
dataset = pd.read_pickle(DATASET_PATH).copy().sort_index()
assert isinstance(dataset.index, pd.DatetimeIndex), "El indice debe ser DatetimeIndex"
assert "target" in dataset.columns, "Falta target"
assert "rp_lag_0" in dataset.columns, "Falta rp_lag_0"
assert "vol_20" in dataset.columns, "Falta vol_20"
assert "vol_60" in dataset.columns, "Falta vol_60"

# Senales del modelo 2 mas una seleccion corta de regimen/persistencia.
eps = 1e-8
abs_r = dataset["rp_lag_0"].abs()
vol_5 = dataset["rp_lag_0"].rolling(5).std(ddof=0)
vol_10 = dataset["rp_lag_0"].rolling(10).std(ddof=0)
vol_20_realized = dataset["rp_lag_0"].rolling(20).std(ddof=0)
vol_20_ref = dataset["vol_20"].abs()
vol_60_ref = dataset["vol_60"].abs()

# Base del mejor MLP anterior.
dataset["vol_5_ratio"] = vol_5 / (vol_20_realized + eps)
dataset["vol_10_ratio"] = vol_10 / (vol_20_realized + eps)
dataset["shock_1"] = abs_r / (vol_20_realized + eps)
dataset["shock_5"] = abs_r.rolling(5).max() / (vol_20_realized + eps)

# Version contenida: se excluyen vol_20_ratio_252, abs_ret_sum_10_ratio y max_abs_ret_10_ratio.
dataset["vol_20_ratio_60"] = vol_20_ref / (vol_60_ref + eps)
dataset["abs_ret_sum_5_ratio"] = abs_r.rolling(5).sum() / (vol_20_realized + eps)

dataset = dataset.replace([np.inf, -np.inf], np.nan).dropna().copy()

feature_cols = [c for c in dataset.columns if c != "target"]
new_features = [
    "vol_5_ratio", "vol_10_ratio", "shock_1", "shock_5",
    "vol_20_ratio_60", "abs_ret_sum_5_ratio",
]

print("Dataset:", dataset.shape)
print("Rango:", dataset.index.min().date(), "->", dataset.index.max().date())
print("Features:", len(feature_cols))
print("Nuevas features incluidas:", [c for c in new_features if c in feature_cols])
print("target mean/std/min/max:", dataset["target"].mean(), dataset["target"].std(), dataset["target"].min(), dataset["target"].max())

assert len(feature_cols) == 28, f"Se esperaban 28 features, hay {len(feature_cols)}"
assert all(c in feature_cols for c in new_features), "Faltan features nuevas"
assert dataset["target"].abs().quantile(0.999) < 0.2, "El target parece tener escala inesperada"


Dataset: (4912, 29)
Rango: 2005-04-29 -> 2024-12-27
Features: 28
Nuevas features incluidas: ['vol_5_ratio', 'vol_10_ratio', 'shock_1', 'shock_5', 'vol_20_ratio_60', 'abs_ret_sum_5_ratio']
target mean/std/min/max: -0.0001957865683106607 0.0074329464425199 -0.07361457194412012 0.0784301570334365


## Metricas de backtesting

In [4]:
def hit_series(real_losses, var_preds):
    real_losses = np.asarray(real_losses, dtype=float).reshape(-1)
    var_preds = np.asarray(var_preds, dtype=float).reshape(-1)
    m = np.isfinite(real_losses) & np.isfinite(var_preds)
    return real_losses[m], var_preds[m], (real_losses[m] > var_preds[m]).astype(int)


def tick_loss(real_losses, var_preds, alpha=0.99):
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    under = np.maximum(real_losses - var_preds, 0.0)
    over = np.maximum(var_preds - real_losses, 0.0)
    return float(np.mean(alpha * under + (1 - alpha) * over))


def winkler_score(real_losses, var_preds, alpha=0.99):
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    exceedance = np.maximum(real_losses - var_preds, 0.0)
    return float(np.mean(var_preds + (2.0 / alpha) * exceedance))


def kupiec_test(real_losses, var_preds, alpha=0.99):
    _, _, I = hit_series(real_losses, var_preds)
    T, x = len(I), int(I.sum())
    p = 1 - alpha
    if T == 0 or x == 0 or x == T:
        return {"LRuc": np.nan, "p_uc": np.nan}
    p_hat = x / T
    LRuc = -2 * np.log(((1 - p) ** (T - x) * p ** x) / ((1 - p_hat) ** (T - x) * p_hat ** x))
    return {"LRuc": LRuc, "p_uc": 1 - chi2.cdf(LRuc, df=1)}


def christoffersen_cc_test(real_losses, var_preds, alpha=0.99):
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    if T < 2:
        return {"LRind": np.nan, "p_ind": np.nan, "LRcc": np.nan, "p_cc": np.nan}
    n00 = n01 = n10 = n11 = 0
    for t in range(1, T):
        pair = (I[t - 1], I[t])
        if pair == (0, 0):
            n00 += 1
        elif pair == (0, 1):
            n01 += 1
        elif pair == (1, 0):
            n10 += 1
        else:
            n11 += 1
    if (n00 + n01) == 0 or (n10 + n11) == 0:
        LRind, p_ind = np.nan, np.nan
    else:
        pi01 = n01 / (n00 + n01)
        pi11 = n11 / (n10 + n11)
        pi = (n01 + n11) / (n00 + n01 + n10 + n11)
        L0 = ((1 - pi) ** (n00 + n10)) * (pi ** (n01 + n11))
        L1 = ((1 - pi01) ** n00) * (pi01 ** n01) * ((1 - pi11) ** n10) * (pi11 ** n11)
        if L0 <= 0 or L1 <= 0:
            LRind, p_ind = np.nan, np.nan
        else:
            LRind = -2 * np.log(L0 / L1)
            p_ind = 1 - chi2.cdf(LRind, df=1)
    kup = kupiec_test(real_losses, var_preds, alpha=alpha)
    LRuc = kup["LRuc"]
    if np.isnan(LRuc) or np.isnan(LRind):
        return {"LRind": LRind, "p_ind": p_ind, "LRcc": np.nan, "p_cc": np.nan}
    LRcc = LRuc + LRind
    return {"LRind": LRind, "p_ind": p_ind, "LRcc": LRcc, "p_cc": 1 - chi2.cdf(LRcc, df=2)}


def dq_test(real_losses, var_preds, alpha=0.99, lags=4):
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    p = 1 - alpha
    if T <= lags + 5:
        return {"DQ": np.nan, "p_dq": np.nan}
    hit = I - p
    y = hit[lags:]
    X = np.column_stack([np.ones(len(y))] + [hit[lags - j:T - j] for j in range(1, lags + 1)])
    XtX = X.T @ X
    if np.linalg.matrix_rank(XtX) < XtX.shape[0]:
        return {"DQ": np.nan, "p_dq": np.nan}
    beta = np.linalg.inv(XtX) @ X.T @ y
    resid = y - X @ beta
    sigma2 = (resid @ resid) / len(y)
    if sigma2 <= 0:
        return {"DQ": np.nan, "p_dq": np.nan}
    DQ = float((beta.T @ XtX @ beta) / sigma2)
    return {"DQ": DQ, "p_dq": 1 - chi2.cdf(DQ, df=X.shape[1])}

In [5]:
def describe_scale(name, x):
    x = np.asarray(x, dtype=float)
    print(f"\n{name}")
    print("min:", np.nanmin(x))
    print("max:", np.nanmax(x))
    print("mean:", np.nanmean(x))
    print("p95:", np.nanpercentile(x, 95))
    print("p99:", np.nanpercentile(x, 99))
    print("p99.9:", np.nanpercentile(x, 99.9))


def plausibility_report(df, var_col="VaR_pred", loss_col="loss_real"):
    describe_scale("loss_real", df[loss_col].values)
    describe_scale(var_col, df[var_col].values)
    n_negative = int((df[var_col] < 0).sum())
    max_date = df[var_col].idxmax()
    min_date = df[var_col].idxmin()
    print("n_negative_var:", n_negative)
    print("max VaR:", max_date, float(df.loc[max_date, var_col]))
    print("min VaR:", min_date, float(df.loc[min_date, var_col]))


def evaluate_var_model(df, model_label, alpha=0.99, sig=0.05):
    real_losses = df["loss_real"].values
    var_preds = df["VaR_pred"].values
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    violations = int(I.sum())
    violation_rate = violations / T
    kup = kupiec_test(real_losses, var_preds, alpha=alpha)
    cc = christoffersen_cc_test(real_losses, var_preds, alpha=alpha)
    dq = dq_test(real_losses, var_preds, alpha=alpha, lags=4)
    df_2020 = df.loc["2020"] if (df.index.year == 2020).any() else pd.DataFrame(columns=df.columns)
    row = {
        "Model": model_label,
        "w": W_LOSS,
        "T": T,
        "Expected Viol.": (1 - alpha) * T,
        "Violations": violations,
        "Violation Rate": violation_rate,
        "VR Gap": abs(violation_rate - (1 - alpha)),
        "Coverage Bias": violation_rate - (1 - alpha),
        "Tick Loss": tick_loss(real_losses, var_preds, alpha=alpha),
        "Winkler Score": winkler_score(real_losses, var_preds, alpha=alpha),
        "Mean VaR Level": float(np.nanmean(var_preds)),
        "Max VaR Level": float(np.nanmax(var_preds)),
        "Min VaR Level": float(np.nanmin(var_preds)),
        "Mean VaR 2020": float(df_2020["VaR_pred"].mean()) if len(df_2020) else np.nan,
        "Max VaR 2020": float(df_2020["VaR_pred"].max()) if len(df_2020) else np.nan,
        "n_negative_var": int((df["VaR_pred"] < 0).sum()),
        "p_UC": kup["p_uc"],
        "UC Test": "PASS" if kup["p_uc"] > sig else "FAIL",
        "p_CC": cc["p_cc"],
        "CC Test": "PASS" if cc["p_cc"] > sig else "FAIL",
        "p_DQ": dq["p_dq"],
        "DQ Test": "PASS" if dq["p_dq"] > sig else "FAIL",
    }
    vals = [row["p_UC"], row["p_CC"], row["p_DQ"]]
    row["p_Mean"] = float(np.mean([v for v in vals if pd.notnull(v)]))
    row["Valid(CC&DQ)"] = "YES" if row["CC Test"] == "PASS" and row["DQ Test"] == "PASS" else "NO"
    return pd.DataFrame([row])


def evaluate_by_year(df, model_label, alpha=0.99):
    rows = []
    for year, g in df.groupby("year"):
        real_losses = g["loss_real"].values
        var_preds = g["VaR_pred"].values
        violations = int(g["viol"].sum())
        expected = (1 - alpha) * len(g)
        kup = kupiec_test(real_losses, var_preds, alpha=alpha)
        cc = christoffersen_cc_test(real_losses, var_preds, alpha=alpha)
        dq = dq_test(real_losses, var_preds, alpha=alpha, lags=4)
        rows.append({
            "Model": model_label,
            "year": int(year),
            "T": len(g),
            "Expected Viol.": expected,
            "Violations": violations,
            "Violation Rate": violations / len(g),
            "VR Gap": abs((violations / len(g)) - (1 - alpha)),
            "Tick Loss": tick_loss(real_losses, var_preds, alpha=alpha),
            "Winkler Score": winkler_score(real_losses, var_preds, alpha=alpha),
            "Mean VaR Level": float(np.nanmean(var_preds)),
            "Max VaR Level": float(np.nanmax(var_preds)),
            "Min VaR Level": float(np.nanmin(var_preds)),
            "Max Loss": float(np.nanmax(real_losses)),
            "Mean Loss": float(np.nanmean(real_losses)),
            "n_negative_var": int((g["VaR_pred"] < 0).sum()),
            "p_UC": kup["p_uc"],
            "UC Test": "PASS" if kup["p_uc"] > SIG else "FAIL",
            "p_CC": cc["p_cc"],
            "CC Test": "PASS" if cc["p_cc"] > SIG else "FAIL",
            "p_DQ": dq["p_dq"],
            "DQ Test": "PASS" if dq["p_dq"] > SIG else "FAIL",
        })
    return pd.DataFrame(rows)


def violation_table(df):
    cols = ["VaR_pred", "loss_real", "viol", "year"]
    out = df.loc[df["viol"] == 1, cols].copy()
    out["exceedance"] = out["loss_real"] - out["VaR_pred"]
    out["exceedance_ratio"] = out["loss_real"] / out["VaR_pred"]
    return out.sort_index()


def worst_days_table(df, n=25):
    out = df.copy()
    out["exceedance"] = out["loss_real"] - out["VaR_pred"]
    out["loss_minus_var"] = out["exceedance"]
    return out.sort_values("loss_real", ascending=False).head(n)


def violation_gap_summary(df):
    v = df.loc[df["viol"] == 1].copy()
    if len(v) < 2:
        return {"violations": len(v), "median_gap_days": np.nan, "gaps_le_5": 0, "gaps_le_20": 0}
    gaps = pd.Series(v.index).diff().dt.days.dropna()
    return {
        "violations": len(v),
        "median_gap_days": float(gaps.median()),
        "gaps_le_5": int((gaps <= 5).sum()),
        "gaps_le_20": int((gaps <= 20).sum()),
    }

## Modelos y entrenamiento

In [6]:
def inverse_softplus(y):
    return math.log(math.expm1(y))


# IMPORTANTE: no modificar. Misma perdida que los experimentos Softplus anteriores.
def var_loss(y_true, y_pred, q=0.99, w=1.0):
    e = y_true - y_pred
    weight = torch.where(e > 0, torch.as_tensor(w, dtype=y_pred.dtype, device=y_pred.device), torch.ones_like(e))
    pinball = torch.maximum(q * e, (q - 1) * e)
    return torch.mean(weight * pinball)


class MLPVaRSoftplusSiLUConservative(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(d_in, 64),
            nn.LayerNorm(64),
            nn.SiLU(),
            nn.Dropout(0.05),
            nn.Linear(64, 32),
            nn.LayerNorm(32),
            nn.SiLU(),
            nn.Dropout(0.05),
            nn.Linear(32, 1),
        )
        self.softplus = nn.Softplus()
        nn.init.constant_(self.body[-1].bias, inverse_softplus(0.02))

    def forward(self, x):
        return self.softplus(self.body(x))


def build_model(d_in, architecture):
    if architecture == "silu_conservative":
        return MLPVaRSoftplusSiLUConservative(d_in=d_in)
    raise ValueError(f"Arquitectura no soportada: {architecture}")


def train_one_model(X_train, y_train, d_in, seed, w_loss, architecture, alpha=0.99, max_epochs=200, lr=5e-5, batch_size=256, patience=15, val_split=0.10):
    torch.manual_seed(seed)
    np.random.seed(seed)
    n = len(X_train)
    split = int(n * (1 - val_split))
    X_tr, X_val = X_train[:split], X_train[split:]
    y_tr, y_val = y_train[:split], y_train[split:]
    model = build_model(d_in=d_in, architecture=architecture)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    train_loader = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)), batch_size=batch_size, shuffle=False)
    X_val_t = torch.from_numpy(X_val)
    y_val_t = torch.from_numpy(y_val)
    best_loss = np.inf
    best_state = None
    patience_counter = 0
    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            opt.zero_grad()
            pred = model(xb)
            loss_val = var_loss(yb, pred, q=alpha, w=w_loss)
            loss_val.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t)
            val_loss = var_loss(y_val_t, val_pred, q=alpha, w=w_loss).item()
        if val_loss < best_loss - 1e-4:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model


## Ejecucion de experimentos

In [7]:
def run_experiment(exp):
    eval_dates = dataset.loc[pd.Timestamp(EVAL_START):pd.Timestamp(EVAL_END)].index
    var_preds = []
    real_targets = []
    dates = []
    current_model = None
    muX, sdX = None, None

    desc = f"{exp['key']} w=1"
    for i, current_date in enumerate(tqdm(eval_dates, desc=desc, dynamic_ncols=True)):
        if i % RETRAIN_EVERY == 0:
            train_end = current_date - pd.Timedelta(days=1)
            train_df = dataset.loc[:train_end].tail(WINDOW)
            if len(train_df) < 250:
                if current_model is None:
                    continue
            else:
                X_train = train_df[feature_cols].values.astype(np.float32)
                y_train = train_df["target"].values.astype(np.float32).reshape(-1, 1)
                muX = X_train.mean(axis=0, keepdims=True)
                sdX = X_train.std(axis=0, keepdims=True) + 1e-8
                X_train = (X_train - muX) / sdX
                current_model = train_one_model(
                    X_train,
                    y_train,
                    d_in=X_train.shape[1],
                    seed=SEED,
                    w_loss=W_LOSS,
                    architecture=exp["architecture"],
                    alpha=ALPHA,
                )

        if current_model is None or muX is None or sdX is None:
            continue

        test_df = dataset.loc[[current_date]]
        X_test = (test_df[feature_cols].values.astype(np.float32) - muX) / sdX
        y_test = test_df["target"].values.astype(np.float32).reshape(-1, 1)

        current_model.eval()
        with torch.no_grad():
            pred = current_model(torch.from_numpy(X_test)).numpy().reshape(-1)[0]

        var_preds.append(float(pred))
        real_targets.append(float(y_test.reshape(-1)[0]))
        dates.append(current_date)

    pred_df = pd.DataFrame({"VaR_pred": var_preds, "loss_real": real_targets}, index=pd.DatetimeIndex(dates)).sort_index()
    pred_df = pred_df.loc[EVAL_START:EVAL_END].dropna().copy()
    pred_df["viol"] = (pred_df["loss_real"] > pred_df["VaR_pred"]).astype(int)
    pred_df["year"] = pred_df.index.year
    return pred_df


results = {}
summary_parts = []
yearly_parts = []
gap_parts = []

for exp in EXPERIMENTS:
    print(f"\n{'=' * 90}\nEjecutando: {exp['label']}\n{'=' * 90}")
    pred_df = run_experiment(exp)
    results[exp["key"]] = pred_df

    summary = evaluate_var_model(pred_df, model_label=exp["label"], alpha=ALPHA, sig=SIG)
    yearly = evaluate_by_year(pred_df, model_label=exp["label"], alpha=ALPHA)
    violations = violation_table(pred_df)
    gaps = violation_gap_summary(pred_df)
    gaps["Model"] = exp["label"]

    pred_path = DATA_DIR / exp["pred_file"]
    summary_path = DATA_DIR / f"{exp['prefix']}_summary.csv"
    yearly_path = DATA_DIR / f"{exp['prefix']}_yearly.csv"
    violations_path = DATA_DIR / f"{exp['prefix']}_violations.csv"

    pred_df.to_csv(pred_path)
    summary.to_csv(summary_path, index=False)
    yearly.to_csv(yearly_path, index=False)
    violations.to_csv(violations_path)

    print("Guardado:", pred_path)
    print("Guardado:", summary_path)
    print("Guardado:", yearly_path)
    print("Guardado:", violations_path)
    plausibility_report(pred_df)
    display(summary)
    display(yearly)
    display(violations)

    summary_parts.append(summary)
    yearly_parts.append(yearly)
    gap_parts.append(gaps)

new_summary = pd.concat(summary_parts, ignore_index=True)
new_yearly = pd.concat(yearly_parts, ignore_index=True)
gap_summary = pd.DataFrame(gap_parts)


Ejecutando: Softplus + conservative dynamic SiLU LayerNorm


silu_conservative w=1: 100%|██████████| 3762/3762 [18:28<00:00,  3.39it/s]

Guardado: ..\data\nn_softplus_silu_conservative_2010_2024_w1.csv
Guardado: ..\data\softplus_silu_conservative_2010_2024_w1_summary.csv
Guardado: ..\data\softplus_silu_conservative_2010_2024_w1_yearly.csv
Guardado: ..\data\softplus_silu_conservative_2010_2024_w1_violations.csv

loss_real
min: -0.07361457496881485
max: 0.07843015342950821
mean: -0.00016187175334081243
p95: 0.009858225146308528
p99: 0.017429363690316665
p99.9: 0.038501730956141275

VaR_pred
min: 0.009949923492968082
max: 0.06314221769571304
mean: 0.025220867868756467
p95: 0.04083549976348877
p99: 0.04774029575288295
p99.9: 0.05507677376642877
n_negative_var: 0
max VaR: 2023-07-20 00:00:00 0.06314221769571304
min VaR: 2015-11-13 00:00:00 0.009949923492968082


,Model,w,T,Expected Viol.,Violations,Violation Rate,VR Gap,Coverage Bias,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Mean VaR 2020,Max VaR 2020,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test,p_Mean,Valid(CC&DQ)
0,Softplus + conservative dynamic SiLU LayerNorm,1,3762,37.62,30,0.007974,0.002026,-0.002026,0.000339,0.025393,0.025221,0.063142,0.00995,0.021302,0.042439,0,0.195554,PASS,0.032386,FAIL,0.0,FAIL,0.07598,NO


,Model,year,T,Expected Viol.,Violations,Violation Rate,VR Gap,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Max Loss,Mean Loss,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test
0,Softplus + conservative dynamic SiLU LayerNorm,2010,252,2.52,1,0.003968,0.006032,0.000269,0.025591,0.025573,0.059406,0.011623,0.026197,-0.000381,0,0.273177,PASS,0.546423,PASS,8.185397e-01,PASS
1,Softplus + conservative dynamic SiLU LayerNorm,2011,252,2.52,4,0.015873,0.005873,0.000366,0.026276,0.026070,0.055922,0.010349,0.026702,-0.000277,0,0.388038,PASS,0.645764,PASS,8.051175e-03,FAIL
2,Softplus + conservative dynamic SiLU LayerNorm,2012,249,2.49,0,0.000000,0.010000,0.000267,0.026595,0.026595,0.049270,0.011940,0.019407,-0.000117,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
3,Softplus + conservative dynamic SiLU LayerNorm,2013,251,2.51,0,0.000000,0.010000,0.000277,0.027802,0.027802,0.052172,0.011639,0.029816,0.000066,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
4,Softplus + conservative dynamic SiLU LayerNorm,2014,252,2.52,0,0.000000,0.010000,0.000295,0.029888,0.029888,0.049961,0.013028,0.025232,0.000438,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
5,Softplus + conservative dynamic SiLU LayerNorm,2015,252,2.52,3,0.011905,0.001905,0.000221,0.020308,0.020261,0.039150,0.009950,0.019679,0.000456,0,0.767969,PASS,0.923289,PASS,9.985401e-01,PASS
6,Softplus + conservative dynamic SiLU LayerNorm,2016,250,2.50,0,0.000000,0.010000,0.000225,0.022125,0.022125,0.046845,0.009998,0.016855,-0.000410,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
7,Softplus + conservative dynamic SiLU LayerNorm,2017,249,2.49,0,0.000000,0.010000,0.000287,0.028247,0.028247,0.054811,0.010914,0.014269,-0.000486,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
8,Softplus + conservative dynamic SiLU LayerNorm,2018,250,2.50,2,0.008000,0.002000,0.000277,0.025745,0.025699,0.049927,0.010907,0.018189,0.000311,0,0.741933,PASS,0.932010,PASS,9.993642e-01,PASS
9,Softplus + conservative dynamic SiLU LayerNorm,2019,251,2.51,1,0.003984,0.006016,0.000248,0.022885,0.022859,0.047256,0.010078,0.018302,-0.000611,0,0.275614,PASS,0.549739,PASS,8.228040e-01,PASS


,VaR_pred,loss_real,viol,year,exceedance,exceedance_ratio
2010-02-03,0.023923,0.026197,1,2010,0.002273,1.095027
2011-09-21,0.014280,0.026702,1,2011,0.012422,1.869845
2011-09-27,0.014091,0.015219,1,2011,0.001128,1.080058
2011-11-16,0.014516,0.017640,1,2011,0.003124,1.215207
2011-12-13,0.014667,0.023733,1,2011,0.009066,1.618100
2015-03-05,0.015061,0.015789,1,2015,0.000728,1.048315
2015-04-07,0.015775,0.016571,1,2015,0.000796,1.050469
2015-08-21,0.013206,0.017541,1,2015,0.004335,1.328274
2018-11-19,0.013010,0.015748,1,2018,0.002739,1.210519
2018-12-21,0.015210,0.018189,1,2018,0.002979,1.195879


## Comparacion final

In [8]:
def load_prediction_csv(path, var_col="VaR_pred"):
    df = pd.read_csv(path, index_col=0, parse_dates=True).sort_index()
    df = df.loc[EVAL_START:EVAL_END].copy()
    if var_col != "VaR_pred":
        df = df.rename(columns={var_col: "VaR_pred"})
    if "viol" not in df.columns:
        df["viol"] = (df["loss_real"] > df["VaR_pred"]).astype(int)
    if "year" not in df.columns:
        df["year"] = df.index.year
    return df

comparison_rows = []
gap_rows = []

reference_files = [
    ("Softplus base w=1", DATA_DIR / "nn_softplus_validation_w1.csv", DATA_DIR / "nn_softplus_test_w1.csv"),
    ("Softplus + ratios + shocks", DATA_DIR / "nn_softplus_pruebas_shocks_2010_2024_w1.csv", None),
    ("Softplus + dynamic features + SiLU LayerNorm", DATA_DIR / "nn_softplus_dynamic_silu_layernorm_2010_2024_w1.csv", None),
    ("Softplus + downside shocks + stress memory", DATA_DIR / "nn_softplus_pruebas_downside_stress_2010_2024_w1.csv", None),
]

for label, path_a, path_b in reference_files:
    if not path_a.exists():
        continue
    if path_b is not None and path_b.exists():
        ref_df = pd.concat([load_prediction_csv(path_a), load_prediction_csv(path_b)]).sort_index()
        ref_df = ref_df[~ref_df.index.duplicated(keep="last")].copy()
        ref_df = ref_df.loc[EVAL_START:EVAL_END]
    else:
        ref_df = load_prediction_csv(path_a)
    comparison_rows.append(evaluate_var_model(ref_df, model_label=label, alpha=ALPHA, sig=SIG))
    gaps = violation_gap_summary(ref_df)
    gaps["Model"] = label
    gap_rows.append(gaps)

comparison_rows.append(new_summary)
gap_rows.extend(gap_parts)

comparison = pd.concat(comparison_rows, ignore_index=True)
comparison_path = DATA_DIR / "softplus_silu_conservative_2010_2024_w1_comparison.csv"
comparison.to_csv(comparison_path, index=False)

gap_comparison = pd.DataFrame(gap_rows)
gap_path = DATA_DIR / "softplus_silu_conservative_2010_2024_w1_gap_comparison.csv"
gap_comparison.to_csv(gap_path, index=False)

print("Guardado:", comparison_path)
print("Guardado:", gap_path)

cols = [
    "Model", "w", "T", "Expected Viol.", "Violations", "Violation Rate", "VR Gap", "Coverage Bias",
    "Tick Loss", "Winkler Score", "Mean VaR Level", "Max VaR Level", "Min VaR Level",
    "Mean VaR 2020", "Max VaR 2020", "n_negative_var", "p_UC", "UC Test", "p_CC", "CC Test", "p_DQ", "DQ Test", "p_Mean", "Valid(CC&DQ)",
]

display(comparison[cols].sort_values(["DQ Test", "CC Test", "UC Test", "VR Gap", "Tick Loss"], ascending=[False, False, False, True, True]))

print("\nResumen de gaps entre violaciones")
display(gap_comparison[["Model", "violations", "median_gap_days", "gaps_le_5", "gaps_le_20"]])

print("\nComparacion anual de los nuevos experimentos")
display(new_yearly.sort_values(["Model", "year"]))

print("\nAnos problematicos de los nuevos experimentos")
display(new_yearly.sort_values(["Violation Rate", "Violations"], ascending=False).head(20))

Guardado: ..\data\softplus_silu_conservative_2010_2024_w1_comparison.csv
Guardado: ..\data\softplus_silu_conservative_2010_2024_w1_gap_comparison.csv


,Model,w,T,Expected Viol.,Violations,Violation Rate,VR Gap,Coverage Bias,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Mean VaR 2020,Max VaR 2020,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test,p_Mean,Valid(CC&DQ)
2,Softplus + dynamic features + SiLU LayerNorm,1,3762,37.62,42,0.011164,0.001164,0.001164,0.000302,0.022348,0.022189,0.070849,0.006577,0.027367,0.067910,0,0.481090,PASS,0.618056,PASS,0.005804,FAIL,0.368316,NO
1,Softplus + ratios + shocks,1,3762,37.62,32,0.008506,0.001494,-0.001494,0.000274,0.018858,0.018684,0.028376,0.008985,0.019036,0.028376,0,0.344590,PASS,0.060605,PASS,0.000000,FAIL,0.135065,NO
0,Softplus base w=1,1,3762,37.62,36,0.009569,0.000431,-0.000431,0.000283,0.018417,0.018217,0.030089,0.007403,0.017326,0.030089,0,0.789181,PASS,0.016972,FAIL,0.000000,FAIL,0.268718,NO
4,Softplus + conservative dynamic SiLU LayerNorm,1,3762,37.62,30,0.007974,0.002026,-0.002026,0.000339,0.025393,0.025221,0.063142,0.009950,0.021302,0.042439,0,0.195554,PASS,0.032386,FAIL,0.000000,FAIL,0.075980,NO
3,Softplus + downside shocks + stress memory,1,3762,37.62,25,0.006645,0.003355,-0.003355,0.000283,0.020356,0.020195,0.030874,0.008058,0.020204,0.030874,0,0.027651,FAIL,0.032905,FAIL,0.000000,FAIL,0.020185,NO



Resumen de gaps entre violaciones


,Model,violations,median_gap_days,gaps_le_5,gaps_le_20
0,Softplus base w=1,36,35.0,8,16
1,Softplus + ratios + shocks,32,53.0,8,12
2,Softplus + dynamic features + SiLU LayerNorm,42,68.0,5,11
3,Softplus + downside shocks + stress memory,25,54.5,6,10
4,Softplus + conservative dynamic SiLU LayerNorm,30,32.0,6,13



Comparacion anual de los nuevos experimentos


,Model,year,T,Expected Viol.,Violations,Violation Rate,VR Gap,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Max Loss,Mean Loss,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test
0,Softplus + conservative dynamic SiLU LayerNorm,2010,252,2.52,1,0.003968,0.006032,0.000269,0.025591,0.025573,0.059406,0.011623,0.026197,-0.000381,0,0.273177,PASS,0.546423,PASS,8.185397e-01,PASS
1,Softplus + conservative dynamic SiLU LayerNorm,2011,252,2.52,4,0.015873,0.005873,0.000366,0.026276,0.026070,0.055922,0.010349,0.026702,-0.000277,0,0.388038,PASS,0.645764,PASS,8.051175e-03,FAIL
2,Softplus + conservative dynamic SiLU LayerNorm,2012,249,2.49,0,0.000000,0.010000,0.000267,0.026595,0.026595,0.049270,0.011940,0.019407,-0.000117,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
3,Softplus + conservative dynamic SiLU LayerNorm,2013,251,2.51,0,0.000000,0.010000,0.000277,0.027802,0.027802,0.052172,0.011639,0.029816,0.000066,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
4,Softplus + conservative dynamic SiLU LayerNorm,2014,252,2.52,0,0.000000,0.010000,0.000295,0.029888,0.029888,0.049961,0.013028,0.025232,0.000438,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
5,Softplus + conservative dynamic SiLU LayerNorm,2015,252,2.52,3,0.011905,0.001905,0.000221,0.020308,0.020261,0.039150,0.009950,0.019679,0.000456,0,0.767969,PASS,0.923289,PASS,9.985401e-01,PASS
6,Softplus + conservative dynamic SiLU LayerNorm,2016,250,2.50,0,0.000000,0.010000,0.000225,0.022125,0.022125,0.046845,0.009998,0.016855,-0.000410,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
7,Softplus + conservative dynamic SiLU LayerNorm,2017,249,2.49,0,0.000000,0.010000,0.000287,0.028247,0.028247,0.054811,0.010914,0.014269,-0.000486,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
8,Softplus + conservative dynamic SiLU LayerNorm,2018,250,2.50,2,0.008000,0.002000,0.000277,0.025745,0.025699,0.049927,0.010907,0.018189,0.000311,0,0.741933,PASS,0.932010,PASS,9.993642e-01,PASS
9,Softplus + conservative dynamic SiLU LayerNorm,2019,251,2.51,1,0.003984,0.006016,0.000248,0.022885,0.022859,0.047256,0.010078,0.018302,-0.000611,0,0.275614,PASS,0.549739,PASS,8.228040e-01,PASS



Anos problematicos de los nuevos experimentos


,Model,year,T,Expected Viol.,Violations,Violation Rate,VR Gap,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Max Loss,Mean Loss,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test
10,Softplus + conservative dynamic SiLU LayerNorm,2020,251,2.51,13,0.051793,0.041793,0.001203,0.023287,0.021302,0.042439,0.010618,0.078430,-0.000757,0,0.000002,FAIL,0.000005,FAIL,2.248403e-09,FAIL
12,Softplus + conservative dynamic SiLU LayerNorm,2022,251,2.51,5,0.019920,0.009920,0.000292,0.022675,0.022534,0.052110,0.010796,0.026797,0.000322,0,0.164040,PASS,0.342892,PASS,8.813249e-01,PASS
1,Softplus + conservative dynamic SiLU LayerNorm,2011,252,2.52,4,0.015873,0.005873,0.000366,0.026276,0.026070,0.055922,0.010349,0.026702,-0.000277,0,0.388038,PASS,0.645764,PASS,8.051175e-03,FAIL
5,Softplus + conservative dynamic SiLU LayerNorm,2015,252,2.52,3,0.011905,0.001905,0.000221,0.020308,0.020261,0.039150,0.009950,0.019679,0.000456,0,0.767969,PASS,0.923289,PASS,9.985401e-01,PASS
8,Softplus + conservative dynamic SiLU LayerNorm,2018,250,2.50,2,0.008000,0.002000,0.000277,0.025745,0.025699,0.049927,0.010907,0.018189,0.000311,0,0.741933,PASS,0.932010,PASS,9.993642e-01,PASS
9,Softplus + conservative dynamic SiLU LayerNorm,2019,251,2.51,1,0.003984,0.006016,0.000248,0.022885,0.022859,0.047256,0.010078,0.018302,-0.000611,0,0.275614,PASS,0.549739,PASS,8.228040e-01,PASS
0,Softplus + conservative dynamic SiLU LayerNorm,2010,252,2.52,1,0.003968,0.006032,0.000269,0.025591,0.025573,0.059406,0.011623,0.026197,-0.000381,0,0.273177,PASS,0.546423,PASS,8.185397e-01,PASS
11,Softplus + conservative dynamic SiLU LayerNorm,2021,252,2.52,1,0.003968,0.006032,0.000317,0.025976,0.025866,0.051511,0.012578,0.030524,-0.000423,0,0.273177,PASS,0.546423,PASS,8.185397e-01,PASS
2,Softplus + conservative dynamic SiLU LayerNorm,2012,249,2.49,0,0.000000,0.010000,0.000267,0.026595,0.026595,0.049270,0.011940,0.019407,-0.000117,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
3,Softplus + conservative dynamic SiLU LayerNorm,2013,251,2.51,0,0.000000,0.010000,0.000277,0.027802,0.027802,0.052172,0.011639,0.029816,0.000066,0,NaN,FAIL,NaN,FAIL,NaN,FAIL


## Lectura esperada

Esta prueba intenta conservar lo bueno del SiLU anterior pero reduciendo su exceso de flexibilidad:

- objetivo principal: subir `p_DQ` hasta `PASS` manteniendo `UC` y `CC` en `PASS`;
- la tasa de violacion deberia quedar cerca del 1%;
- `Max VaR Level` deberia bajar claramente frente al SiLU anterior;
- si `DQ` no pasa, revisar si mejora `p_DQ` sin empeorar mucho `Tick Loss` ni cobertura;
- mirar especialmente 2012, 2018, 2020 y 2021.
